# Importing packages 

In [1]:
import os
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
from mplcursors import cursor
import seaborn as sns
import csv
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio  

# Default renderer of pio is plotly_mimetype+notebook, but jekyll fails to 
# parse plotly_mimetype.
pio.renderers.default = 'notebook_connected'
import statsmodels 

# Load the require.js library for plotly to work in jekyll, which is required 
# for interactive plots.
from IPython.display import display, HTML
js = '<script src="https://cdnjs.cloudflare.com/ajax/libs/require.js/2.3.6/' \
'require.min.js" integrity="sha512-c3Nl8+7g4LMSTdrm621y7kf9v3SDPnhxLNhc' \
'jFJbKECVnmZHTdo+IRO05sNLTH/D3vA6u1X32ehoLC7WFVdheg==" ' \
'crossorigin="anonymous"></script>'
display(HTML(js))

# Import the custom modules  
import hts.core as hts
import hts.variables as hts_var

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
# Suppress SettingWithCopyWarning
import warnings
warnings.filterwarnings('ignore', category=pd.errors.SettingWithCopyWarning)

<span style="font-family: 'Arial'; font-size: 20px; color: #ffffff;">
Importing and cleaning of the primary raw data
</span>

In [2]:
# Create a dictionary that maps each folder path to a specific ID number
# The data is one level up from the notebooks.
folder_dict = {
    '1': '../data/plate1',
    '2': '../data/plate2',
    '3': '../data/plate3',
    '4': '../data/plate4',
    '5': '../data/plate5'
}

df = hts.load_plates(folder_dict)

# Number of rows and columns of the dataframe
print(f"DataFrame shape: {df.shape}")

# Display the first 5 rows of the dataframe to check if it loaded correctly
df.head(5)

DataFrame shape: (1540, 289)


,Row,Column,Timepoint,Nuclei in confirmed total cell region - Number of Objects,Nuclei in confirmed total cell region - Nucleus Area [µm²] - Mean per Well,Nuclei in confirmed total cell region - Nucleus Roundness - Mean per Well,Nuclei in confirmed total cell region - Nucleus Perimeter [µm] - Mean per Well,Nuclei in confirmed total cell region - Nucleus Width [µm] - Mean per Well,Nuclei in confirmed total cell region - Nucleus Length [µm] - Mean per Well,Nuclei in confirmed total cell region - Nucleus Ratio Width to Length - Mean per Well,Nuclei in confirmed total cell region - Nucleus HOECHST 33342 Haralick Correlation 1 px - Mean per Well,Nuclei in confirmed total cell region - Nucleus HOECHST 33342 Haralick Contrast 1 px - Mean per Well,Nuclei in confirmed total cell region - Nucleus HOECHST 33342 Haralick Sum Variance 1 px - Mean per Well,Nuclei in confirmed total cell region - Nucleus HOECHST 33342 Haralick Homogeneity 1 px - Mean per Well,Nuclei in confirmed total cell region - Nucleus HOECHST 33342 SER Spot 0 px - Mean per Well,Nuclei in confirmed total cell region - Nucleus HOECHST 33342 SER Hole 0 px - Mean per Well,Nuclei in confirmed total cell region - Nucleus HOECHST 33342 SER Edge 0 px - Mean per Well,Nuclei in confirmed total cell region - Nucleus HOECHST 33342 SER Ridge 0 px - Mean per Well,Nuclei in confirmed total cell region - Nucleus HOECHST 33342 SER Valley 0 px - Mean per Well,Nuclei in confirmed total cell region - Nucleus HOECHST 33342 SER Saddle 0 px - Mean per Well,Nuclei in confirmed total cell region - Nucleus HOECHST 33342 SER Bright 0 px - Mean per Well,Nuclei in confirmed total cell region - Nucleus HOECHST 33342 SER Dark 0 px - Mean per Well,Nuclei in confirmed total cell region - Intensity Nucleus HOECHST 33342 Mean - Mean per Well,Nuclei in confirmed total cell region - Intensity Nucleus HOECHST 33342 StdDev - Mean per Well,Nuclei in confirmed total cell region - Intensity Nucleus HOECHST 33342 Median - Mean per Well,Nuclei in confirmed total cell region - Intensity Nucleus HOECHST 33342 Sum - Mean per Well,Nuclei in confirmed total cell region - Individual cells Selected - Mean per Well,Confirmed total cell region selected - Number of Nuclei in confirmed total cell region (2)- per Object - Sum per Well,Confirmed total cell region selected - Confirmed total cell region selected Area [µm²] - Sum per Well,Confirmed total cell region selected - Number of Spots - Sum per Well,Individual cells Selected - Number of Objects,Individual cells Selected - Nucleus Area [µm²] - Mean per Well,Individual cells Selected - Nucleus Roundness - Mean per Well,Individual cells Selected - Nucleus Perimeter [µm] - Mean per Well,Individual cells Selected - Nucleus Width [µm] - Mean per Well,Individual cells Selected - Nucleus Length [µm] - Mean per Well,Individual cells Selected - Nucleus Ratio Width to Length - Mean per Well,Individual cells Selected - Nucleus HOECHST 33342 Haralick Correlation 1 px - Mean per Well,Individual cells Selected - Nucleus HOECHST 33342 Haralick Contrast 1 px - Mean per Well,Individual cells Selected - Nucleus HOECHST 33342 Haralick Sum Variance 1 px - Mean per Well,Individual cells Selected - Nucleus HOECHST 33342 Haralick Homogeneity 1 px - Mean per Well,Individual cells Selected - Nucleus HOECHST 33342 SER Spot 0 px - Mean per Well,Individual cells Selected - Nucleus HOECHST 33342 SER Hole 0 px - Mean per Well,Individual cells Selected - Nucleus HOECHST 33342 SER Edge 0 px - Mean per Well,Individual cells Selected - Nucleus HOECHST 33342 SER Ridge 0 px - Mean per Well,Individual cells Selected - Nucleus HOECHST 33342 SER Valley 0 px - Mean per Well,Individual cells Selected - Nucleus HOECHST 33342 SER Saddle 0 px - Mean per Well,Individual cells Selected - Nucleus HOECHST 33342 SER Bright 0 px - Mean per Well,Individual cells Selected - Nucleus HOECHST 33342 SER Dark 0 px - Mean per Well,Individual cells Selected - Intensity Nucleus HOECHST 33342 Mean - Mean per Well,Individual cells S

##### Making an ID for plates and wells

In [ ]:
# Map the numbering of the rows to actual alphabet as seen on the plate. 
hts.map_num_to_letter(df, col='Row', inplace=True)

# Convert the col of df called "Column" to str so that we can combine several
# strings in the next step.
df.Column = df.Column.astype(str)

# Combine the plate number and the row and column of the plate to make a 
# unique id for each well.
df['Plate format'] = df[
    ['File ID', 'Row', 'Column']
].apply(
        lambda x: ''.join(x), axis=1
)

   
<span style="font-family: 'Arial'; font-size: 20px; color: #ffffff;">
Delete the unnecessary columns and resort the columns
</span>

In [ ]:
all_nan_columns = df.columns[
    df.eq('NaN').all()
].tolist()

useless_cols = all_nan_columns + [
    "Row", "Column", "Timepoint", 
    "Time [s]", "Compound", "Concentration", 
    "Cell Type", "Cell Count", "Unnamed: 287", 
    "Number of Analyzed Fields"
]

# Delete the columns that have no value for the analysis
df.drop(columns=useless_cols, inplace=True)

In [ ]:
# Change the location of the columns to make it more readable and easier to 
# analyze. The columns are moved based on the order of importance for the 
# analysis.
df.insert(0, "Plate format", 
          df.pop("Plate format")
)

df.insert(1, "File ID", 
          df.pop("File ID")
)

df.insert(2, "Individual cells Selected - Number of Objects", 
          df.pop("Individual cells Selected - Number of Objects")
)

df.insert(
    3, "Confirmed total cell region selected - " \
    "Confirmed total cell region selected Area [µm²] - Sum per Well", 
    df.pop("Confirmed total cell region selected - " \
    "Confirmed total cell region selected Area [µm²] - Sum per Well")
)

df.insert(4, "Confirmed total cell region selected - Number of Spots - " \
    "Sum per Well", 
    df.pop("Confirmed total cell region selected - Number of Spots - " \
    "Sum per Well")
)
